# 第 5 章 · Tool Schema 与模型输出解析

**这一章你会得到什么**：搞清楚模型“想执行一条命令”这件事，是怎么从一个 tool call 变成 Environment 能吃的统一 `action` 的；以及**在进 Environment 之前**，哪些非法输出会被 `FormatError` 拦下。

> 这是 Model 边界的核心职责：provider-specific 的解析，绝不外泄给 Agent。

In [1]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

mini-SWE-agent: 2.4.5


In [5]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 实验 1：看 Bash Tool Schema 长什么样

这是发给模型 API 的“工具说明书”，告诉模型可以调用一个叫 `bash`、带 `command` 参数的函数。

In [4]:
from pprint import pprint
from minisweagent.models.utils.actions_toolcall import BASH_TOOL
pprint(BASH_TOOL)

{'function': {'description': 'Execute a bash command',
              'name': 'bash',
              'parameters': {'properties': {'command': {'description': 'The '
                                                                       'bash '
                                                                       'command '
                                                                       'to '
                                                                       'execute',
                                                        'type': 'string'}},
                             'required': ['command'],
                             'type': 'object'}},
 'type': 'function'}


## 实验 2：把一个合法 tool call 解析成 action

`parse_toolcall_actions` 只依赖 tool call 的三个属性：`.id`、`.function.name`、`.function.arguments`。我们用 `SimpleNamespace` 造一个假的。

In [6]:
from types import SimpleNamespace
def tool_call(command="echo hello", name="bash", call_id="call_1", arguments=None):
    if arguments is None:
        import json
        arguments = json.dumps({"command": command})
    return SimpleNamespace(id=call_id, function=SimpleNamespace(name=name, arguments=arguments))

In [7]:
from minisweagent.models.utils.actions_toolcall import parse_toolcall_actions
actions = parse_toolcall_actions(
    [tool_call(command="ls -la")],
    format_error_template="{{ error }}",
)
print(actions)

[{'command': 'ls -la', 'tool_call_id': 'call_1'}]


## 观察点：统一的 action 形状

解析结果是 `{"command": "ls -la", "tool_call_id": "call_1"}`。
Environment 完全不知道它来自哪家 API，只读 `action["command"]`。**上游能变，下游接口不变**——这就是好边界。

## 实验 3：一次解析多个 tool call

模型一轮可以请求多条命令。解析器会返回一个 action 列表，`tool_call_id` 各自独立。

In [8]:
many = parse_toolcall_actions(
    [tool_call(command="pwd", call_id="c1"), tool_call(command="whoami", call_id="c2")],
    format_error_template="{{ error }}",
)
for a in many:
    print(a)

{'command': 'pwd', 'tool_call_id': 'c1'}
{'command': 'whoami', 'tool_call_id': 'c2'}


## 实验 4：观察三种 `FormatError`

模型输出不合法时，解析器**在进入 Environment 之前**就拒绝。跑一下，看三种典型错误的提示。

In [9]:
from minisweagent.exceptions import FormatError
cases = {
    "没有工具调用": [],
    "未知工具":   [tool_call(name="python")],
    "非法 JSON":  [tool_call(arguments="{not-json")],
    "缺少 command": [tool_call(arguments='{"cmd": "ls"}')],
}
for label, calls in cases.items():
    try:
        parse_toolcall_actions(calls, format_error_template="{{ error }}")
    except FormatError as e:
        print(f"[{label}] {e.messages[0]['content']}")

[没有工具调用] No tool calls found in the response. Every response MUST include at least one tool call.
[未知工具] Unknown tool 'python'.
[非法 JSON] Error parsing tool call arguments: Expecting property name enclosed in double quotes: line 1 column 2 (char 1).Missing 'command' argument in bash tool call.
[缺少 command] Missing 'command' argument in bash tool call.


## 动手：预测每种错误的 role 与 interrupt_type

不运行下面这格，先猜：`FormatError` 携带的消息，`role` 是什么？`extra.interrupt_type` 是什么？
写下答案再运行核对。（提示：格式错误是要“喂回给模型让它改正”的。）

In [13]:
try:
    parse_toolcall_actions([tool_call(arguments="{not-json")], format_error_template="ERROR: {{ error }}")
except FormatError as e:
    print(e.messages[0])

{'role': 'user', 'content': "ERROR: Error parsing tool call arguments: Expecting property name enclosed in double quotes: line 1 column 2 (char 1).Missing 'command' argument in bash tool call.", 'extra': {'interrupt_type': 'FormatError'}}


## 观察点
- 纠错消息的 `role` 是 `user`——因为它要作为“用户反馈”喂回模型，让模型重新生成合法 tool call。
- `extra.interrupt_type == "FormatError"`，`run()` 靠它区分“该重试”还是“该退出”（回忆第 4 章的异常家族）。
- 解析失败**不能**把原始字符串直接丢给 Environment：那等于让未经校验的模型输出直达 shell，既不安全也无法保证有 `command`。

## 连回真实源码

In [15]:
show_source("src/minisweagent/models/utils/actions_toolcall.py", 30, 76)

30  def parse_toolcall_actions(
31      tool_calls: list, *, format_error_template: str, template_kwargs: dict | None = None
32  ) -> list[dict]:
33      """Parse tool calls from the response. Raises FormatError if unknown tool or invalid args.
34  
35      ``template_kwargs`` are extra variables exposed to ``format_error_template`` (e.g.
36      ``{"finish_reason": ...}`` so a template can distinguish a real format mistake from a
37      ``max_tokens`` truncation).
38      """
39      template_kwargs = template_kwargs or {}
40      if not tool_calls:
41          raise FormatError(
42              {
43                  "role": "user",
44                  "content": Template(format_error_template, undefined=StrictUndefined).render(
45                      error="No tool calls found in the response. Every response MUST include at least one tool call.",
46                      actions=[],
47                      has_tool_calls=False,
48                      **template_kwargs,
49          

## 闭卷检查
1. 为什么 action parsing 属于 Model 边界而不是 Agent？
2. 统一 action 的两个字段是什么？Environment 读哪个？
3. 三种 FormatError 分别在校验哪一步？